In [1]:
import tensorflow as tf
import TensorSlider as ts
import keras
import numpy as np
import DataPrep


2025-02-12 14:39:31.092734: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1739367571.103571   77695 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1739367571.106612   77695 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-02-12 14:39:31.118955: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
tfrecordpath = "../Data/tfrecords/"

windowsize = 150
lookahead = 5
batch_size = 20

coins = ["BTCUSD_PERP", "ETHUSD_PERP", "ADAUSD_PERP", "SOLUSD_PERP", "BNBUSDT_PERP", "LINKUSD_PERP", "TRXUSD_PERP", "XLMUSD_PERP","DOTUSD_PERP"]
datasets = DataPrep.getAllSliders(coins, windowsize, lookahead, batch_size, tfrecordpath)
valdataset = datasets.pop(-1)


I0000 00:00:1739367574.033729   77695 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5592 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060 Ti, pci bus id: 0000:05:00.0, compute capability: 8.6


In [3]:
import keras
import os

def load_model(name, optimizer):
    """
    Load model and latest checkpoint (if applicable). Returns model, and last epoch that was trained.
    If no checkpoints present, either creates the checkpoint folder or trains directly on the saved model.
    """
    folder = "models/" + name + "/"
    # Load model
    model = keras.models.load_model(folder + "model.keras")

    checkpoint = tf.train.Checkpoint(optimizer=optimizer, model=model)
    manager = tf.train.CheckpointManager(checkpoint=checkpoint, directory=folder, max_to_keep=3)
    manager.restore_or_initialize()

    return model, manager

class saveEachEpoch(tf.keras.callbacks.Callback):
    """
    custom callback for saving checkpoints because of course we need to do this on our own
    """

    def __init__(self, checkpointmanager):
        super().__init__()
        # no idea if we want to or need to super this
        try:
            self.lastEpoch = int(checkpointmanager.latest_checkpoint.split("-")[-1])
        except Exception as e:
            print(e)
            self.lastEpoch = 0

        self.checkpointManager = checkpointmanager
        print("Model was trained for " + str(self.lastEpoch) + " epochs before.")

    def on_epoch_end(self, epoch, logs):
        """
        Create a checkpoint and save.
        """
        print(f"Epoch {self.lastEpoch} ended")
        # Increment which epoch this is
        self.lastEpoch += 1
        # Save weights
        self.checkpointManager.save(checkpoint_number=epoch)


In [4]:
#zip the different dataset sources
zipped = tf.data.Dataset.zip(datasets=tuple(datasets))
def combineZippedBatches(*zipped):
    batchshape = zipped[0][0]
    # Create first tensors to concat the rest
    data = zipped[0][0]
    label = zipped[0][1]
    for i in range(1, len(zipped)): # iterate over each remaining pair
        data = tf.concat([data, zipped[i][0]], axis=0)
        label = tf.concat([label, zipped[i][1]], axis=0)

    return data, label

@tf.function
def duplicateLabels(data, label):
    stretchamount = 3
    copied = []
    for i in range(stretchamount):
        copied.append(label)

    return data, tuple(copied)

# combine batches into one megabatch
batchTogether = zipped.map(combineZippedBatches, num_parallel_calls=tf.data.AUTOTUNE)
trainingset = batchTogether.map(DataPrep.createLabelsBatch, num_parallel_calls=tf.data.AUTOTUNE)

# duplicate labels
#trainingset = trainingset.map(duplicateLabels, num_parallel_calls=tf.data.AUTOTUNE)

valset = valdataset.map(DataPrep.createLabelsBatch)
#valset = valset.map(duplicateLabels, num_parallel_calls=tf.data.AUTOTUNE)

# Prefetch and create labels
training = trainingset.prefetch(tf.data.AUTOTUNE)
validation = valset.prefetch(tf.data.AUTOTUNE)

## Custom losses and metrics

In [5]:
def minmax_MSE(y_true, y_pred):
    return tf.reduce_mean(tf.square(y_true[:, 0:2] - y_pred[:,0:2]), axis=1)

def expected_MSE(y_true, y_pred):
    # Calculate mse of expected log variance
    return tf.reduce_mean(tf.square(y_pred[:,2:4]), axis=1)

def error_of_error_MSE(y_true, y_pred):
    return tf.abs(minmax_MSE(y_true, y_pred) - expected_MSE(y_true, y_pred))

def expected_error_variance(y_true, y_pred):
    exerrors = tf.reduce_mean(y_pred[:,2:4], axis=0)
    return tf.math.reduce_variance(exerrors)

def decision_loss(y_true, y_pred):

    minprofit = 0.07 # We want at least 1 percent profit, remember 1 = 10%


    # Check if the real minimum is below predicted min + error, and vice versa
    stoppedlong = tf.less_equal(y_true[:,0], y_pred[:,0] - y_pred[:,2])
    stoppedshort = tf.greater_equal(y_true[:,1], y_pred[:,1] + y_pred[:,3])

    # Check if real maximum is above predicted max - error, and vice versa
    longtp = tf.greater_equal(y_true[:,1], y_pred[:,1] - y_pred[:,3])
    shorttp = tf.less_equal(y_true[:,0], y_pred[:,0] + y_pred[:,2])

    # Compute predicted price delta
    # delta = max-error - min+error
    pricedelta = (y_pred[:,1] - y_pred[:,3]) - (y_pred[:,0] + y_pred[:,2])

    # Check if pricedelta is profitable
    profitable = tf.greater(pricedelta, minprofit)

    # long/short only true if stopped is false, it reaches take profit, and the trade would be profitable
    long = tf.logical_and(tf.logical_and(tf.logical_not(stoppedlong), profitable), longtp)
    short = tf.logical_and(tf.logical_and(tf.logical_not(stoppedshort), profitable), shorttp)

    # if both long and short, check which entry price closer. Entry price is min/max prediction
    conflicting = tf.math.logical_and(long, short)
    distancelong = tf.abs(y_true[:,0])
    distanceshort = tf.abs(y_true[:,1])

    # Where we have a conflict, pick the side closer to opening price
    resolvelong = tf.where(conflicting, tf.less_equal(distancelong, distanceshort), tf.greater(distancelong, distanceshort))
    resolveshort = tf.where(conflicting, tf.greater(distancelong, distanceshort), tf.less_equal(distancelong, distanceshort))

    # if long and conflict long, true, if long and conflict false, false, if only one true, true, so it's an OR
    resultlong = tf.math.logical_or(long, resolvelong)
    resultshort = tf.math.logical_or(short, resolveshort)

    # neutral if neither long nor short
    resultneutral = tf.logical_not(tf.logical_or(resultlong, resultshort))

    result = tf.cast(tf.stack([resultlong, resultshort, resultneutral], axis=1), tf.int32)
    cross_entropy = keras.losses.CategoricalCrossentropy(reduction=None)
    cross_entropy_loss = cross_entropy(result, y_pred[:,4:7])
    return cross_entropy_loss

def profitabletrades(y_true, y_pred):
    minprofit = 0.15 # We want at least 1.5 percent profit, remember 1 = 10%

    # Check if the real minimum is below predicted min + error, and vice versa
    stoppedlong = tf.less_equal(y_true[:,0], y_pred[:,0] - y_pred[:,2])
    stoppedshort = tf.greater_equal(y_true[:,1], y_pred[:,1] + y_pred[:,3])

    # Check if real maximum is above predicted max - error, and vice versa
    longtp = tf.greater_equal(y_true[:,1], y_pred[:,1] - y_pred[:,3])
    shorttp = tf.less_equal(y_true[:,0], y_pred[:,0] + y_pred[:,2])

    # Compute predicted price delta
    # delta = max-error - min+error
    pricedelta = (y_pred[:,1] - y_pred[:,3]) - (y_pred[:,0] + y_pred[:,2])

    # Check if pricedelta is profitable
    profitable = tf.greater(pricedelta, minprofit)

    # long/short only true if stopped is false, it reaches take profit, and the trade would be profitable
    long = tf.logical_and(tf.logical_and(tf.logical_not(stoppedlong), profitable), longtp)
    short = tf.logical_and(tf.logical_and(tf.logical_not(stoppedshort), profitable), shorttp)

    # if both long and short, check which entry price closer. Entry price is min/max prediction
    conflicting = tf.math.logical_and(long, short)
    distancelong = tf.abs(y_true[:,0])
    distanceshort = tf.abs(y_true[:,1])

    # Where we have a conflict, pick the side closer to opening price
    resolvelong = tf.where(conflicting, tf.less_equal(distancelong, distanceshort), tf.greater(distancelong, distanceshort))
    resolveshort = tf.where(conflicting, tf.greater(distancelong, distanceshort), tf.less_equal(distancelong, distanceshort))

    # if long and conflict long, true, if long and conflict false, true, if not long and not conflict, false, so an OR
    resultlong = tf.math.logical_or(long, resolvelong)
    resultshort = tf.math.logical_or(short, resolveshort)

    # neutral if neither long nor short
    resultneutral = tf.logical_not(tf.logical_or(resultlong, resultshort))


    result = tf.cast(tf.stack([resultlong, resultshort, resultneutral], axis=1), tf.int32)

    # now we have what should be true, lets check what is actually true
    # the truth is y_pred[;,4:7]
    # prediction is argmax of result
    trade = tf.argmax(y_pred[:,4:7], axis=1) # which index is predicted
    correct = tf.argmax(result, axis=1) # which index is correct
    profits =tf.where(trade == correct, 1, -1) # if index is correct, return 1, else return -1
    profits = tf.where(trade == 2, 0, profits) # if the index is 2, i.e. neutral, return 0
    #return tf.cast(tf.reduce_mean(profits), tf.float32)
    return tf.cast(profits, tf.float32)

def amountNeutral(y_true, y_pred):
    return y_pred[:,6]

def isNeutral(y_true, y_pred):
    neutral = tf.argmax(y_pred[:,4:7], axis=1)
    return tf.where(neutral == 2, 1, 0)

def combinedloss(y_true, y_pred):

    minmaxMSE = minmax_MSE(y_true, y_pred)
    errorMSE = error_of_error_MSE(y_true, y_pred)
    decisionloss = decision_loss(y_true, y_pred)
    neutral = amountNeutral(y_true, y_pred) # incentivize less neutrality
    profits = tf.math.sigmoid(1 - profitabletrades(y_true, y_pred)) # higher proportion profitable trades -> less loss

    summed_loss = minmaxMSE + errorMSE + decisionloss + tf.square(profits) + tf.square(neutral)
    stacked_loss = tf.stack([minmaxMSE, tf.square(errorMSE), decisionloss, tf.square(profits)])
    return stacked_loss


In [ ]:
modelName = "dingus4"

tensorboard = keras.callbacks.TensorBoard(
                                                log_dir=f"models/{modelName}/logs",
                                                histogram_freq=100,
                                                write_graph=True,
                                                write_images=False,
                                                write_steps_per_second=True,
                                                update_freq="batch",
                                                #profile_batch = '70,100',
                                                embeddings_freq=0,
                                                embeddings_metadata=None,
                                            )
optimizer = keras.optimizers.Adam(amsgrad=True, clipvalue=0.2)

model, manager = load_model(modelName, optimizer)

model.compile(loss=combinedloss,
                optimizer=optimizer,
                metrics=[expected_error_variance, minmax_MSE, error_of_error_MSE, decision_loss, amountNeutral, profitabletrades, isNeutral],
              )

model.summary()

history = model.fit(training, epochs=10, verbose=0, validation_data=validation, callbacks=[saveEachEpoch(manager), tensorboard])

/root/miniconda3/envs/levbot/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'adam', because it has 56 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 6, 5, 150) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 4500)      │          0 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 6000)      │ 27,006,000 │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 6000)      │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 6000)      │ 36,006,000 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 6000)      │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 4000)      │ 24,004,000 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 4000)      │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 4000)      │ 16,004,000 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 4000)      │          0 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 1000)      │  4,001,000 │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 5000)      │  5,005,000 │ dense_4[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ minmax (Dense)      │ (None, 2)         │     10,002 │ dense_5[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ error (Dense)       │ (None, 2)         │     10,002 │ dense_5[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decision (Dense)    │ (None, 3)         │     15,003 │ dense_5[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 7)         │          0 │ minmax[0][0],     │
│ (Concatenate)       │                   │            │ error[0][0],      │
│                     │                   │            │ decision[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 112,061,007 (427.48 MB)

 Trainable params: 112,061,007 (427.48 MB)

 Non-trainable params: 0 (0.00 B)

Model was trained for 0 epochs before.
